# [Video RAG] 프레임 캡션처리

In [ ]:
# 구글드라이브 연동
from google.colab import drive
drive.mount('/content/drive')  # 내 구글드라이브를 /content/drive 경로로 마운트

BASE_PATH = '/content/drive/MyDrive/SK네트웍스 Family AI 캠프 23기/05_multimodal_rag'  # 기본작업경로 변수

In [ ]:
import os
from google.colab import userdata  # Colab 비밀키 저장소 접근 모듈

# 환경변수 설정
os.environ['LANGSMITH_TRACING'] = 'true'  # LangSmith 추적 기능 활성화
os.environ['LANGSMITH_ENDPOINT'] = 'https://api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'skn23-multimodal-rag'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
import base64
from openai import OpenAI
from PIL import Image
import os

# 이미지 파일을 base64 문자열로 반환하는 함수
def encode_image_to_base64(image_path):
    with open(image_path, 'rb') as f:
        return base64.b64encode(f.read()).decode('utf-8')  # base64 인코딩 후 문자열로 변환

# OpenAI 멀티모달 모델을 호출해 이미지 설명을 생성하는 함수
def call_openai(client, model, image_b64):
    # Chat Completions API 호출
    response = client.chat.completions.create(
        model = model,
        messages=[  # 시스템/유저 메시지 구성
            {
                'role': 'system',
                'content': '''당신은 이미지에 대한 설명을 간략하고 정확하게 작성하는 **장면해설가**입니다.'''  # 역할 설정
            },
            {
                'role': 'user',
                'content': [
                    {
                        'type': 'text',
                        'text': '''아래 이미지는 동영사의 한 프레임입니다. 프레임의 주요특징과 장면을 간략하고 정확하게 설명해주세요.'''  # 텍스트 지시문
                    },
                    {
                        'type': 'image_url',
                        'image_url': {
                            'url': f'data:image/jpeg;base64,{image_b64}'  # base64 이미지 전달
                        }
                    }
                ]
            }
        ],
        temperature = 0.2,  # 출력 다양성 (일관적인 응답)
        max_completion_tokens = 1024  # 최대 생성 토큰 수
    )

    return response.choices[0].message.content

client = OpenAI()
model = 'gpt-4.1-mini'
image_path = os.path.join(BASE_PATH, 'frames', 'skiing', 'skiing_frame00180.jpg')
image_b64 = encode_image_to_base64(image_path)

image = Image.open(image_path)  # 이미지 객체 열기
image.thumbnail((480, 270))     # 화면 출력 크기 조정
display(image)

print(call_openai(client, model, image_b64))

## 전체이미지 캡셔닝

In [ ]:
import re  # 정규표현식 모듈

# 프레임 파일명에서 숫자 프레임 번호를 정수로 추출하는 함수
def extract_frame_no(frame_file):
    match = re.search(r'_frame(\d+)\.jpg$', frame_file)  # '_frame' 뒤 숫자를 캡처해서 .jpg로 끝나는지 확인
    return int(match.group(1))  # 캡처된 숫자 문자열을 int로 변환해 반환

extract_frame_no('alpinist_frame00300.jpg')

In [ ]:
# 비디오별 프레임 이미지 캡션 생성 후 CSV 저장
import uuid  # 고유 ID 부여 모듈
import pandas as pd
from tqdm import tqdm
import os

# frames 폴더의 각 비디오 프레임을 순회하면서 이미지 설명을 생성하고 CSV 파일로 저장하는 함수
def generate_frame_image_captions(frame_dir='frames', video_dir='videos', caption_dir='captions'):

    # 경로 설정
    frame_dir_path = os.path.join(BASE_PATH, frame_dir)
    video_dir_path = os.path.join(BASE_PATH, video_dir)
    caption_dir_path = os.path.join(BASE_PATH, caption_dir)
    os.makedirs(caption_dir_path, exist_ok=True)  # 폴더 생성

    # 비디오 이름 불러오기
    video_names = [d for d in os.listdir(frame_dir_path)]

    # 비디오별 프레임 캡션처리
    for video_name in video_names:
        vedio_filename = f'{video_name}.mp4'
        video_frame_dir_path = os.path.join(frame_dir_path, video_name)  # 해당 비디오의 프레임 폴더 경로
        frame_files = [f for f in os.listdir(video_frame_dir_path)]  # 프레임 파일 목록

        results = []

        # 프레임별 캡션 처리
        for frame_file in tqdm(frame_files, desc=f'{video_name} 프레임 해설 중'):
            frame_file_path = os.path.join(video_frame_dir_path, frame_file)  # 프레임 파일 경로
            image_b64 = encode_image_to_base64(frame_file_path)  # 프레임 파일 이미지 -> base64
            description = call_openai(client, model, image_b64)
            frame_no = extract_frame_no(frame_file)  # 파일명 -> 프레임 번호
            results.append({
                'id': str(uuid.uuid4()),  # 각 레코드별 UUID 생성
                'video_filename': vedio_filename,
                'frame_no': frame_no,
                'description': description
            })

        # 비디오별 프레임 설명 DataFrame -> csv로 저장
        df = pd.DataFrame(results)
        csv_file_path = os.path.join(caption_dir_path, f'{video_name}_frames.csv')
        df.to_csv(csv_file_path, index=False, encoding='utf-8')  # CSV 저장(인덱스 제외, 한글깨짐방지용)
        print(f'{video_name}: {len(df)}개 프레임 이미지별 캡션 저장 완료!\n')

generate_frame_image_captions()
